# 2 · Batch — run every image

The tutorial processed one image. Here we run the **same** `segment → measure`
pipeline over all 121 cross-sections and collect two tidy tables in `outputs/`:

- **`all_vessels.csv`** — one row per vessel (every detected vessel in every
  image), with the shape metrics plus `site` / `group` / `rainfall`.
- **`area_fraction.csv`** — one row per image: the fraction of the frame taken
  up by vessel lumens, again tagged with `site` / `group` / `rainfall`.

Notebook 3 reads both of these. Run this notebook once before opening it.

In [1]:
import os
from pathlib import Path

if not Path("data/10x").exists() and Path("../data/10x").exists():
    os.chdir("..")
print("working directory:", Path.cwd())

working directory: C:\Users\metas\repos\vessel_analysis


In [2]:
import cv2
import pandas as pd

import vessel_morphometry as vm

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

### Calibration: pixels → microns

Real-world units come from the **scale bar** on each plate. Run the calibration
script once to measure every bar and write `outputs/calibration.csv`:

```
uv run python scripts/calibrate_dataset.py
```

We then look up each image's **µm/px**. The lookup prefers a hand-entered
`label_um_manual` (÷ `bar_px`), then the script's `um_per_px`, then the cohort
median. If `calibration.csv` is missing it falls back to the constant
`CALIBRATION_UM_PER_PX["10x"]` from `config.py` (which ships as `None`, leaving
`area_um2` as `NaN`) and warns.

Reading the printed labels uses OCR (the optional `[ocr]` extra,
`rapidocr-onnxruntime`); if it is not installed the bar lengths still calibrate,
because the calibration run found µm/px to be the **same across plates** (~1.167)
— so the constant applies to every image.

In [3]:
from vessel_morphometry.calibrate import um_per_px_lookup

# Per-image µm/px from the calibration run; default is used for any image not in
# the table (and the config constant if calibration.csv is missing entirely).
um_per_px, default_um_per_px = um_per_px_lookup(
    "outputs/calibration.csv", fallback_constant=vm.CALIBRATION_UM_PER_PX["10x"])
print("calibrated images:", len(um_per_px), "| default µm/px:", default_um_per_px)

calibrated images: 121 | default µm/px: 1.1628


### Run the pipeline over every image

We loop over the images in sorted order so the output is reproducible, measure
each one, and stash both the per-vessel table and the per-image area fraction.

In [4]:
paths = sorted(Path("data/10x").glob("*.png"))
print(f"{len(paths)} images to process\n")

per_vessel_frames = []
area_fraction_rows = []

LENGTH_PX_COLS = ["diam_major_px", "diam_minor_px", "feret_max_px", "equiv_diam_px"]

for n, path in enumerate(paths, start=1):
    img = cv2.imread(str(path))
    s = um_per_px.get(path.stem, default_um_per_px)   # microns per pixel for this plate
    p = vm.Params(um_per_px=s)
    labels = vm.segment(img, p)
    df = vm.measure(labels, img.shape, p, source=path.stem)
    # measure() fills area_um2 but its output column order omits the linear *_um
    # columns, so add them here (length_px * µm/px) when we have a calibration.
    if s and len(df):
        for px_col in LENGTH_PX_COLS:
            df[px_col.replace("_px", "_um")] = df[px_col] * s
    per_vessel_frames.append(df)

    # one area-fraction row per image, tagged with the experiment metadata
    site, group, rainfall = vm.parse_name(path.stem)
    fraction = vm.vessel_area_fraction(labels)
    area_fraction_rows.append({
        "source_image": path.stem, "site": site, "group": group,
        "rainfall": rainfall, "vessel_fraction": fraction["vessel_fraction"],
        "vessel_px": fraction["vessel_px"], "image_px": fraction["image_px"],
    })

    print(f"[{n:3d}/{len(paths)}] {path.stem:28s} -> {len(df):3d} vessels")

print("\ndone.")

121 images to process

[  1/121] GUNDA_ACC_T20_1_10x          ->  13 vessels
[  2/121] GUNDA_ACC_T20_2_10x          ->   8 vessels


[  3/121] GUNDA_ACC_T20_3_10x          ->  18 vessels


[  4/121] GUNDA_ACC_T4_1_10x           -> 137 vessels


[  5/121] GUNDA_ACC_T4_2_10x           -> 134 vessels


[  6/121] GUNDA_ACC_T4_3_10x           -> 138 vessels


...

[117/121] STURT_SHED_T7_1_10x (2)      ->  98 vessels


[118/121] STURT_SHED_T7_1_10x          -> 105 vessels


[119/121] STURT_SHED_T7_2_10x(2)       ->  98 vessels


[120/121] STURT_SHED_T7_2_10x          -> 141 vessels


[121/121] STURT_SHED_T7_3_10x          -> 101 vessels

done.


### Save the two tables

In [5]:
all_vessels = pd.concat(per_vessel_frames, ignore_index=True)
area_fraction = pd.DataFrame(area_fraction_rows)

all_vessels.to_csv(OUT / "all_vessels.csv", index=False)
area_fraction.to_csv(OUT / "area_fraction.csv", index=False)

print("all_vessels.csv :", all_vessels.shape, "->", OUT / "all_vessels.csv")
print("area_fraction.csv:", area_fraction.shape, "->", OUT / "area_fraction.csv")

all_vessels.csv : (12090, 22) -> outputs\all_vessels.csv
area_fraction.csv: (121, 7) -> outputs\area_fraction.csv


### Quick sanity check

In [6]:
n_images = len(paths)
n_with_vessels = all_vessels["source_image"].nunique()

print("images processed       :", n_images)
print("images with >= 1 vessel:", n_with_vessels,
      f"({n_images - n_with_vessels} detected none)")
print("total vessels          :", len(all_vessels))
print("vessels per image (median):",
      int(all_vessels.groupby("source_image").size().median()))
print("\nvessels by rainfall class:")
print(all_vessels["rainfall"].value_counts())

# area_fraction always has one row per image, even when a section has no vessels.
assert len(area_fraction) == n_images, "area_fraction should cover every image"
all_vessels.head()

images processed       : 121
images with >= 1 vessel: 120 (1 detected none)
total vessels          : 12090
vessels per image (median): 112

vessels by rainfall class:
rainfall
low         5384
moderate    4678
high        2028
Name: count, dtype: int64


,source_image,vessel_id,circularity,area_px,area_um2,diam_major_px,diam_minor_px,feret_max_px,equiv_diam_px,aspect_ratio,...,touches_border,site,group,rainfall,cx,cy,diam_major_um,diam_minor_um,feret_max_um,equiv_diam_um
0,GUNDA_ACC_T20_1_10x,1,0.716808,284.0,393.099239,27.741283,13.539250,30.016662,19.015784,0.488054,...,False,GUNDA,ACC,high,767.109155,26.345070,32.637620,15.928928,35.314603,22.372070
1,GUNDA_ACC_T20_1_10x,2,0.665676,298.0,412.477371,29.278972,14.476040,29.068884,19.478845,0.494418,...,False,GUNDA,ACC,high,743.785235,33.872483,34.446711,17.031061,34.199542,22.916861
2,GUNDA_ACC_T20_1_10x,3,0.347160,538.0,744.673911,30.824108,26.657779,32.310989,26.172560,0.864835,...,False,GUNDA,ACC,high,51.485130,267.671004,36.264563,31.362877,38.013878,30.792016
3,GUNDA_ACC_T20_1_10x,4,0.253270,45.0,62.286851,14.065579,6.552992,14.035669,7.569398,0.465889,...,False,GUNDA,ACC,high,137.000000,276.866667,16.548154,7.709595,16.512964,8.905396
4,GUNDA_ACC_T20_1_10x,5,0.863736,372.0,514.904637,28.414401,16.888068,28.635642,21.763389,0.594349,...,False,GUNDA,ACC,high,121.338710,339.801075,33.429543,19.868812,33.689833,25.604627


### Two things to notice about the counts

- **Zero-vessel images.** An occasional section detects no vessels at all; it
  contributes no rows to `all_vessels.csv`, so that table may cover slightly
  fewer than 121 images, while `area_fraction.csv` always has one row per image.
- **How many vessels?** This run keeps **every** detected vessel (~12,000 total,
  a median of ~112 per image) using the package defaults `min_area_px=30`,
  `max_area_px=4000`. If you want a smaller, cleaner table, raise `min_area_px`
  via `vm.Params(...)` here — that is a tuning knob, not a change to the package
  formulas. Notebook 3 also trims to whole vessels in a sensible size window
  before any statistics.

In [7]:
area_fraction.groupby("site")["vessel_fraction"].describe().round(4)

,count,mean,std,min,25%,50%,75%,max
site,,,,,,,,
GUNDA,24.0,0.0467,0.0379,0.0000,0.0064,0.0556,0.0785,0.1024
NOCO,41.0,0.0548,0.0303,0.0007,0.0436,0.0605,0.0735,0.1116
STURT,56.0,0.0530,0.0277,0.0001,0.0391,0.0559,0.0763,0.1000
